In [60]:
import os
import json

import numpy as np
import pandas as pd

In [61]:
# runs = [
#     os.path.join('experiments', str(x)) for x in range(744, 824)
# ] # without +/- fix in Z_0 calculation

# runs = [
#     os.path.join('experiments', str(x)) for x in range(1154, 1194)
# ] # fixed +/- in Z_0 calculation

# runs = [
#     os.path.join('experiments', str(x)) for x in range(1417, 1457)
# ]

# assert len(runs) == 80, len(runs)

original_runs = [
    os.path.join('experiments', str(x)) for x in range(1522, 1562)
]

runs = [
    os.path.join('experiments', str(x)) for x in range(1562, 1582)
]

for run in original_runs:
    hparams = json.load(open(os.path.join(run, 'hparams.json')))
    if hparams['add_bc'] and not hparams['do_masking']:
        continue
    else:
        runs.append(run)

assert len(runs) == 40, len(runs)

In [62]:
soln_number_to_name = {
    1: 'Plane waves',
    2: 'Radial waves',
    3: 'Hopf fibration',
    4: 'Random solution'
}

In [63]:
df = pd.DataFrame(
    columns=list(json.load(open(os.path.join(runs[0], 'hparams.json'))).keys())
    + [
        'soln_name',
        'min_mse_val',
        'time_to_min_mse_val',
        'min_rel_l2_error',
        'time_to_min_rel_l2_error',
        'time_to_5p_rel_l2_error'
    ]
)

for run in runs:
    info = json.load(open(os.path.join(run, 'hparams.json')))

    assert (info['do_masking'] and info['add_bc']) or (not info['add_bc'])
    
    info['soln_name'] = soln_number_to_name[info['soln']]
    
    log = pd.read_csv(os.path.join(run, 'log.tsv'), sep='\t')
    
    if os.path.exists(os.path.join(run, 'init_time.txt')):
        init_time = float(open(os.path.join(run, 'init_time.txt')).read())
    else:
        init_time = 0.
    
    # print(log['training_time'].cumsum().max())
    
    # epoch where we reach minimum validation MSE
    min_mse_val = log['mse_val'].min()
    info['min_mse_val'] = min_mse_val
    
    min_mse_val_epoch = int(log[log['mse_val'] == min_mse_val].iloc[0]['epoch'])
    # how much training time to get there?
    info['time_to_min_mse_val'] = log[log['epoch'] <= min_mse_val_epoch]['training_time'].sum().item() + init_time
    
    # epoch where we reach minimum validation relative L2 error
    min_rel_l2_error = log['rel_l2_error'].min()
    info['min_rel_l2_error'] = min_rel_l2_error
    
    min_rel_l2_error_epoch = int(log[log['rel_l2_error'] <= min_rel_l2_error].iloc[0]['epoch'])
    info['time_to_min_rel_l2_error'] = log[log['epoch'] <= min_rel_l2_error_epoch]['training_time'].sum().item() + init_time
    
    # epoch where we first reach 5% relative error
    first_5p_epoch = int(log[log['rel_l2_error'] < 0.05].iloc[0]['epoch'])
    info['time_to_5p_rel_l2_error'] = log[log['epoch'] <= first_5p_epoch]['training_time'].sum().item() + init_time
    
    df.loc[len(df)] = info

In [64]:
results = df[[
    'soln_name',
    'time_to_5p_rel_l2_error',
    'time_to_min_rel_l2_error',
    'min_rel_l2_error',
    'add_bc',
    'seed'
]].rename(columns={
    'time_to_5p_rel_l2_error': 'Time to <5% RL2E. (s)',
    'time_to_min_rel_l2_error': 'Time to min. RL2E. (s)',
    'min_rel_l2_error': 'Min. RL2E. (%)',
    'soln_name': 'Solution',
    'add_bc': 'w/ BC'
}).groupby(by=['Solution', 'w/ BC']).agg(['mean', 'sem']).drop(columns=['seed'])

display_results = {}
metrics = results.columns.get_level_values(0).unique()

for col in metrics:
    mean = results[(col, 'mean')]
    sem = results[(col, 'sem')]
    if 'Time' in col:
        display_results[col] = mean.round(1).astype(str) + ' ± ' + sem.round(1).astype(str)
    else:  # relative error columns
        mean *= 100.0
        sem *= 100.0
        display_results[col] = mean.map('{:.2f}'.format) + ' ± ' + sem.map('{:.2f}'.format)

display_results = pd.DataFrame(display_results).reset_index()
display_results['Solution'] = display_results['Solution'].map(
    {'Random solution': 4, 'Plane waves': 1, 'Radial waves': 2, 'Hopf fibration': 3}
)
display_results = display_results.sort_values(by=['Solution', 'w/ BC'], ascending=[True, False])
display_results = display_results[['Solution', 'w/ BC', 'Min. RL2E. (%)', 'Time to <5% RL2E. (s)', 'Time to min. RL2E. (s)']].reset_index(drop=True)
display_results

,Solution,w/ BC,Min. RL2E. (%),Time to <5% RL2E. (s),Time to min. RL2E. (s)
0,1,True,1.03 ± 0.01,8.1 ± 0.4,52.4 ± 5.1
1,1,False,1.09 ± 0.03,6.8 ± 0.3,41.7 ± 3.5
2,2,True,2.89 ± 0.18,58.9 ± 2.0,499.4 ± 23.1
3,2,False,1.01 ± 0.00,11.9 ± 0.6,101.7 ± 4.7
4,3,True,1.00 ± 0.00,7.4 ± 0.3,108.5 ± 15.1
5,3,False,1.16 ± 0.05,3.2 ± 0.0,12.3 ± 0.7
6,4,True,1.68 ± 0.06,58.3 ± 3.2,475.5 ± 4.6
7,4,False,1.04 ± 0.01,6.1 ± 0.2,60.4 ± 2.8


In [65]:
print(display_results.to_markdown(index=False))

|   Solution | w/ BC   | Min. RL2E. (%)   | Time to <5% RL2E. (s)   | Time to min. RL2E. (s)   |
|-----------:|:--------|:-----------------|:------------------------|:-------------------------|
|          1 | True    | 1.03 ± 0.01      | 8.1 ± 0.4               | 52.4 ± 5.1               |
|          1 | False   | 1.09 ± 0.03      | 6.8 ± 0.3               | 41.7 ± 3.5               |
|          2 | True    | 2.89 ± 0.18      | 58.9 ± 2.0              | 499.4 ± 23.1             |
|          2 | False   | 1.01 ± 0.00      | 11.9 ± 0.6              | 101.7 ± 4.7              |
|          3 | True    | 1.00 ± 0.00      | 7.4 ± 0.3               | 108.5 ± 15.1             |
|          3 | False   | 1.16 ± 0.05      | 3.2 ± 0.0               | 12.3 ± 0.7               |
|          4 | True    | 1.68 ± 0.06      | 58.3 ± 3.2              | 475.5 ± 4.6              |
|          4 | False   | 1.04 ± 0.01      | 6.1 ± 0.2               | 60.4 ± 2.8               |
